For the first steps of the pipeline I will be using popglen, so I need to format raw data to match requirements. To make the units and sample files I proceed tot he same method as with ATTW. 

# Popglen launching 
The config yaml file was created and the parameters and reasoning is explained in that obsidian.  

In [ ]:
conda activate popglen
conda config --show channel_priority #check it must be set to strict 
#conda config --set channel_priority strict #if not already set to strict

which mamba #make sure mamba solver is installed ##good 

#launch popglen 
cd /media/ssd/popglen_workflow/

export JAVA_TOOL_OPTIONS="-Djava.awt.headless=true" # this needs to be done first otherwise Qualimap will fail

snakemake   --configfile config/config_tridactylus.yml   --use-conda   --conda-prefix ~/snakemake_conda --conda-frontend conda   --use-singularity   --cores 40   --jobs 40   --default-resources mem_mb=9000

# Bioinformatics

The popglen workflow can't combine ld files, so I will need to do downstream analyses manually. 

## LD decay
I need to calculate decay, in order how much pruning to do. 

### In bash

In [ ]:
#check if there are empty files
find /media/ssd/popglen_workflow/results/datasets/p_tridactylus_10_2026/analyses/ngsLD/chunks/ \
-name "p_tridactylus_10_2026.dataset-ref_all_chunk*_allsites-filts.ld_maxkbdist-4000_rndsample-0.05.gz" \
-size 0c \
| wc -l #there are 4 empty files which are produced in upstream analyses because they didn't pass the 0.05 distance filter of ngsLD 

# create a file with only file paths of ref_all chunks, excluding empty files 
find /media/ssd/popglen_workflow/results/datasets/p_tridactylus_10_2026/analyses/ngsLD/chunks/ \
-name "p_tridactylus_10_2026.dataset-ref_all_chunk*_allsites-filts.ld_maxkbdist-4000_rndsample-0.05.gz" \
-size +0c \
> /media/ssd/Bioinformatics/p_tridactylus_10/ngsLD/LD_decay/ld_file_list.txt

wc -l /media/ssd/Bioinformatics/p_tridactylus_10/ngsLD/LD_decay/ld_file_list.txt #how many lines

# Merge all LD chunks
zcat $(cat /media/ssd/Bioinformatics/p_dorsalis_07/ngsLD/LD_decay/ld_file_list.txt) \
> /media/ssd/Bioinformatics/p_tridactylus_10/ngsLD/all_ref_all.ld

#check how many lines it has, to see if I can downsample it for the plotting of the curve
wc -l /media/ssd/Bioinformatics/p_tridactylus_10/ngsLD/all_ref_all.ld # 7125026 

#downsample to keep 1 000 000 pairs and keep the dist (column3) and r2 (column7) columns
awk '{print $3, $7}' \
/media/ssd/Bioinformatics/p_tridactylus_10/ngsLD/all_ref_all.ld \
| shuf -n 1000000 \
> /media/ssd/Bioinformatics/p_tridactylus_10/ngsLD/LD_decay/ld_sampled.txt

head /media/ssd/Bioinformatics/p_tridactylus_10/ngsLD/LD_decay/ld_sampled.txt

### In R
#### LD decay curve

In [ ]:
cat > /tmp/ld_decay.R <<'EOF'
ld <- read.table(
  "/media/ssd/Bioinformatics/p_tridactylus_10/ngsLD/LD_decay/ld_sampled.txt"
)

colnames(ld) <- c("dist", "r2")

breaks <- seq(0, max(ld$dist), by = 500)

ld$bin <- cut(ld$dist, breaks = breaks)

agg <- aggregate(r2 ~ bin, ld, mean)

midpoints <- (breaks[-1] + breaks[-length(breaks)]) / 2
midpoints <- midpoints[1:nrow(agg)]

pdf(
  "/media/ssd/Bioinformatics/p_tridactylus_10/ngsLD/LD_decay/ld_decay_simple.pdf"
)

plot(
  midpoints,
  agg$r2,
  type = "l",
  xlab = "Distance (bp)",
  ylab = "Mean r2"
)

dev.off()
EOF

Rscript /tmp/ld_decay.R

As seen in the plot, a very steep curve indicates low linkage desiquilbrium. It reaches a plateau at around 25 000-35 000, but to keep consistent with the other species I will set the plateau in the next parameters as 50 000.  
### SNP density 

In [ ]:
cat > /tmp/snp_density.R <<'EOF'
ld <- read.table(
  "/media/ssd/Bioinformatics/p_tridactylus_10/ngsLD/all_ref_all.ld",
  header = FALSE
)

colnames(ld) <- c(
  "site1", "site2", "dist",
  "r2_ExpG", "D", "Dp", "r2"
) # add column names

get_pos <- function(x) {
  as.numeric(sub(".*:", "", x))
}

pos <- c(get_pos(ld$site1), get_pos(ld$site2))

n_snps <- length(unique(pos))
n_snps

contig <- sub(":.*", "", c(ld$site1, ld$site2))
pos <- as.numeric(sub(".*:", "", c(ld$site1, ld$site2)))

df <- data.frame(contig, pos)

library(dplyr)

genome_span <- df %>%
  group_by(contig) %>%
  summarise(span = max(pos) - min(pos)) %>%
  summarise(total = sum(span)) %>%
  pull(total)

density <- n_snps / genome_span

density
EOF

Rscript /tmp/snp_density.R